In [1]:
!pip install -q ucimlrepo
from ucimlrepo import fetch_ucirepo
import os

ds = fetch_ucirepo(id=94)
df = ds.data.features.copy()
df["is_spam"] = ds.data.targets.iloc[:, 0]
os.makedirs("data", exist_ok=True)
df.to_csv("data/spambase.csv", index=False)
print(df.shape)
print(df["is_spam"].value_counts())

(4601, 58)
is_spam
0    2788
1    1813
Name: count, dtype: int64


In [2]:
import pandas as pd, joblib, os
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

SEED = 2026
TARGET = "is_spam"

df = pd.read_csv("data/spambase.csv")
print("duplicates:", df.duplicated().sum(), "| nulls:", df.isnull().sum().sum())
df = df.drop_duplicates()
print("rows after removing duplicates:", len(df))

X, y = df.drop(columns=[TARGET]), df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

models = {
    "logistic_regression": Pipeline([("sc", StandardScaler()),
                            ("clf", LogisticRegression(max_iter=2000, C=1.0))]),
    "decision_tree": Pipeline([("clf", DecisionTreeClassifier(max_depth=7, random_state=SEED))]),
    "knn": Pipeline([("sc", StandardScaler()),
                     ("clf", KNeighborsClassifier(n_neighbors=9))]),
    "naive_bayes": Pipeline([("clf", GaussianNB())]),
    "svm": Pipeline([("sc", StandardScaler()),
                     ("clf", SVC(kernel="rbf", C=3, probability=True, random_state=SEED))]),
}

os.makedirs("model", exist_ok=True)
os.makedirs("results", exist_ok=True)
rows = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, f"model/{name}.joblib")
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    rows.append({"ML Model Name": name,
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, prob),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "MCC": matthews_corrcoef(y_test, pred)})

res = pd.DataFrame(rows).round(4)
print(res.to_string(index=False))
res.to_csv("results/metrics.csv", index=False)

test = X_test.copy(); test[TARGET] = y_test
test.to_csv("test_data.csv", index=False)
print("train/test sizes:", len(X_train), len(X_test))
print("train class counts:\n", y_train.value_counts())
print("test class counts:\n", y_test.value_counts())

import sklearn, pandas, numpy
print("VERSIONS -> scikit-learn", sklearn.__version__, "| pandas", pandas.__version__,
      "| numpy", numpy.__version__, "| joblib", joblib.__version__)

duplicates: 391 | nulls: 0
rows after removing duplicates: 4210
      ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
logistic_regression    0.9097 0.9627     0.9088  0.8601 0.8838 0.8109
      decision_tree    0.9062 0.9335     0.9003  0.8601 0.8798 0.8035
                knn    0.8955 0.9528     0.9079  0.8214 0.8625 0.7811
        naive_bayes    0.8076 0.9405     0.6875  0.9494 0.7975 0.6526
                svm    0.9264 0.9737     0.9281  0.8839 0.9055 0.8459
train/test sizes: 3368 842
train class counts:
 is_spam
0    2025
1    1343
Name: count, dtype: int64
test class counts:
 is_spam
0    506
1    336
Name: count, dtype: int64
VERSIONS -> scikit-learn 1.6.1 | pandas 2.2.3 | numpy 2.4.6 | joblib 1.6.0


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, confusion_matrix,
    ConfusionMatrixDisplay, classification_report)

st.set_page_config(page_title="Spam Mail Detector", page_icon="📧", layout="wide")

TARGET = "is_spam"
MODELS = {
    "Logistic Regression": "logistic_regression",
    "Decision Tree": "decision_tree",
    "KNN": "knn",
    "Naive Bayes": "naive_bayes",
    "Support Vector Machine": "svm",
}

st.title("📧 Spam Mail Detector")
st.write("Pick a saved model, upload test emails (as 57 numeric features plus the "
         "`is_spam` column), and see how well the model separates spam from normal mail. "
         "The models are only loaded, never retrained.")

st.sidebar.header("Step 1: Test data")
uploaded = st.sidebar.file_uploader("Upload test CSV", type="csv")
use_default = st.sidebar.checkbox("Use bundled test_data.csv", value=False)
st.sidebar.header("Step 2: Model")
choice = st.sidebar.selectbox("Select model", list(MODELS))

if uploaded is not None:
    df = pd.read_csv(uploaded)
elif use_default:
    df = pd.read_csv("test_data.csv")
else:
    st.info("Upload test_data.csv in the sidebar (or tick the bundled file) to begin.")
    st.stop()

if TARGET not in df.columns:
    st.error(f"The file must contain the target column '{TARGET}'.")
    st.stop()

model = joblib.load(f"model/{MODELS[choice]}.joblib")
needed = list(model.feature_names_in_)
missing = [c for c in needed if c not in df.columns]
if missing:
    st.error(f"Missing {len(missing)} feature columns, for example: {missing[:5]}")
    st.stop()

X = df[needed]
y = df[TARGET]
pred = model.predict(X)
prob = model.predict_proba(X)[:, 1]

st.subheader(f"Results for {choice} on {len(df)} emails")
c = st.columns(6)
c[0].metric("Accuracy", f"{accuracy_score(y, pred):.4f}")
c[1].metric("AUC", f"{roc_auc_score(y, prob):.4f}")
c[2].metric("Precision", f"{precision_score(y, pred):.4f}")
c[3].metric("Recall", f"{recall_score(y, pred):.4f}")
c[4].metric("F1 Score", f"{f1_score(y, pred):.4f}")
c[5].metric("MCC", f"{matthews_corrcoef(y, pred):.4f}")

left, right = st.columns(2)
with left:
    st.markdown("**Confusion matrix** (0 = not spam, 1 = spam)")
    fig, ax = plt.subplots(figsize=(4, 3.5))
    ConfusionMatrixDisplay(confusion_matrix(y, pred),
                           display_labels=["Not spam", "Spam"]).plot(ax=ax, cmap="Blues")
    st.pyplot(fig)
with right:
    st.markdown("**Classification report**")
    st.code(classification_report(y, pred, target_names=["Not spam", "Spam"]))

with st.expander("Preview of uploaded data"):
    st.dataframe(df.head(20))

Writing app.py


In [4]:
%%writefile requirements.txt
streamlit
scikit-learn==1.6.1
pandas==2.2.3
numpy==2.4.6
joblib==1.6.0
matplotlib

Writing requirements.txt


In [5]:
!zip -r project.zip app.py requirements.txt test_data.csv data model results
from google.colab import files
files.download("project.zip")

  adding: app.py (deflated 54%)
  adding: requirements.txt (deflated 9%)
  adding: test_data.csv (deflated 86%)
  adding: data/ (stored 0%)
  adding: data/spambase.csv (deflated 88%)
  adding: model/ (stored 0%)
  adding: model/logistic_regression.joblib (deflated 30%)
  adding: model/naive_bayes.joblib (deflated 28%)
  adding: model/svm.joblib (deflated 78%)
  adding: model/knn.joblib (deflated 76%)
  adding: model/decision_tree.joblib (deflated 65%)
  adding: results/ (stored 0%)
  adding: results/metrics.csv (deflated 37%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>